# Testing `mps2vec(psi, B)` as a symmetry-sector map

This notebook checks what `mps2vec(psi, B)` actually does.

Conclusion up front:

- If `psi` already lives in the symmetry sector `B`, then `mps2vec(psi, B)` returns the correct sector coefficients.
- For a generic state, it is **not** the full orthogonal projection onto that sector.

The tests below verify both statements for several symmetry combinations.

In [1]:
using LinearAlgebra, Random, ITensors, ITensorMPS

DEV = true
if DEV
    include("../../src/EDKit.jl")
    using .EDKit
else
    using EDKit
end

Random.seed!(7)

TaskLocalRNG()

## Helper functions

For a symmetry basis `B`, we build the embedding matrix `P` from sector coordinates to the full tensor-product basis. If `v_sector` is a vector in the reduced basis, then `P * v_sector` is the corresponding full many-body state.

This lets us test both:

1. round-trip correctness for states already in the sector;
2. comparison with the true orthogonal projection `P' * ψ_full` for a generic state.

In [2]:
orbit_order(B) = B isa EDKit.AbstractOnsiteBasis ? 1 : EDKit.order(B)

function sector_embedding(B::AbstractBasis)
    Bc = B
    full = TensorBasis(L=length(Bc), base=Bc.B)
    P = zeros(ComplexF64, size(full, 1), size(Bc, 1))
    for j in 1:size(full, 1)
        change!(full, j)
        Bc.dgt .= full.dgt
        C, pos = index(Bc)
        iszero(C) || (P[j, pos] = C / orbit_order(Bc))
    end
    P
end

function sector_to_full(v::AbstractVector, B::AbstractBasis)
    sector_embedding(B) * v
end

function roundtrip_error(B::AbstractBasis)
    P = sector_embedding(B)
    v_sector = randn(ComplexF64, size(B, 1)) |> normalize
    ψ_full = P * v_sector
    s = siteinds(B.B, length(B))
    ψ_mps = vec2mps(ψ_full, s)
    v_back = mps2vec(ψ_mps, B)
    norm(v_back - v_sector), norm(P' * ψ_full - v_sector)
end

function projection_mismatch(B::AbstractBasis)
    P = sector_embedding(B)
    ψ_full = randn(ComplexF64, size(P, 1)) |> normalize
    s = siteinds(B.B, length(B))
    ψ_mps = vec2mps(ψ_full, s)
    v_mps = mps2vec(ψ_mps, B)
    v_exact = P' * ψ_full
    norm(v_mps - v_exact)
end

projection_mismatch (generic function with 1 method)

## Round-trip test on states already in the sector

For each basis `B`, we generate a random vector in the reduced sector, embed it into the full Hilbert space, convert it to an MPS, and map it back with `mps2vec(psi, B)`.

In [3]:
L = 6
Nhalf = L ÷ 2

sectors = [
    ("N", basis(L=L, N=Nhalf)),
    ("k", basis(L=L, k=1)),
    ("p", basis(L=L, p=1)),
    ("z", basis(L=L, z=1)),
    ("N+k", basis(L=L, N=Nhalf, k=1)),
    ("N+p", basis(L=L, N=Nhalf, p=1)),
    ("N+z", basis(L=L, N=Nhalf, z=1)),
    ("k+p", basis(L=L, k=0, p=1)),
    ("k+z", basis(L=L, k=1, z=1)),
    ("p+z", basis(L=L, p=1, z=1)),
    ("N+k+p", basis(L=L, N=Nhalf, k=0, p=1)),
    ("N+k+z", basis(L=L, N=Nhalf, k=1, z=1)),
    ("N+p+z", basis(L=L, N=Nhalf, p=1, z=1)),
    ("k+p+z", basis(L=L, k=0, p=1, z=1)),
    ("N+k+p+z", basis(L=L, N=Nhalf, k=0, p=1, z=1)),
]

results = map(sectors) do (name, B)
    err_mps, err_exact = roundtrip_error(B)
    (name=name, dim=size(B, 1), mps2vec_roundtrip=err_mps, exact_projector_roundtrip=err_exact)
end

for row in results
    println(rpad(row.name, 10), " dim=", lpad(row.dim, 4),
            "   |mps2vec - v| = ", row.mps2vec_roundtrip,
            "   |P'Pv - v| = ", row.exact_projector_roundtrip)
end

N          dim=  20   |mps2vec - v| = 3.875001055746515e-16   |P'Pv - v| = 0.0
k          dim=   9   |mps2vec - v| = 1.2093797368966774e-15   |P'Pv - v| = 1.996741323115087e-16
p          dim=  36   |mps2vec - v| = 1.0187743314327377e-15   |P'Pv - v| = 1.6855965334625945e-16
z          dim=  32   |mps2vec - v| = 7.540276163037847e-16   |P'Pv - v| = 1.475465324456903e-16
N+k        dim=   3   |mps2vec - v| = 4.809409384954737e-16   |P'Pv - v| = 1.3877787807814457e-16
N+p        dim=  10   |mps2vec - v| = 3.993671027827731e-16   |P'Pv - v| = 1.9497705388809518e-16
N+z        dim=  10   |mps2vec - v| = 5.481506594438372e-16   |P'Pv - v| = 1.6336822229997407e-16
k+p        dim=  13   |mps2vec - v| = 4.989524313017156e-16   |P'Pv - v| = 2.105818725546248e-16
k+z        dim=   4   |mps2vec - v| = 9.265177410400736e-16   |P'Pv - v| = 1.1274368112207335e-16
p+z        dim=  20   |mps2vec - v| = 5.053364790017857e-16   |P'Pv - v| = 1.0233490192354574e-16
N+k+p      dim=   3   |mps2vec - v| = 3.

If the implementation is doing the sector reconstruction correctly, the `mps2vec` round-trip errors above should be close to machine precision.

## Translation with a larger unit cell

Now test the same idea when translation is defined with a unit cell of size `a = 2`.

In [ ]:
L2 = 6
Nhalf2 = L2 ÷ 2

cell_sectors = [
    ("k,a=2", basis(L=L2, k=1, a=2)),
    ("N+k,a=2", basis(L=L2, N=Nhalf2, k=1, a=2)),
    ("k+p,a=2", basis(L=L2, k=0, p=1, a=2)),
    ("k+z,a=2", basis(L=L2, k=1, z=1, a=2)),
    ("N+k+p,a=2", basis(L=L2, N=Nhalf2, k=0, p=1, a=2)),
]

cell_results = map(cell_sectors) do (name, B)
    err_mps, err_exact = roundtrip_error(B)
    (name=name, dim=size(B, 1), mps2vec_roundtrip=err_mps, exact_projector_roundtrip=err_exact)
end

for row in cell_results
    println(rpad(row.name, 14), " dim=", lpad(row.dim, 4),
            "   |mps2vec - v| = ", row.mps2vec_roundtrip,
            "   |P'Pv - v| = ", row.exact_projector_roundtrip)
end

These enlarged-unit-cell translation sectors should again show round-trip errors near machine precision.

## Compare against the true orthogonal projection for a generic state

Now we test a state with no reason to lie in the sector. We compare:

- `mps2vec(psi, B)`
- the exact orthogonal projection coefficients `P' * ψ_full`

If `mps2vec(psi, B)` were a generic projector, these would match. They generally should not.

In [4]:
Btest = basis(L=L, N=Nhalf, k=0, p=1)
projection_mismatch(Btest)

0.5410223781514562

A nonzero value here means `mps2vec(psi, B)` is **not** the same as the orthogonal projector on a generic state.

That is consistent with the implementation: it reads amplitudes on symmetry representatives and reconstructs sector coefficients assuming the state already obeys the symmetry relations.

## Symmetry-preserving eigenstate example

As one more check, take an eigenstate computed directly inside a symmetry sector, embed it into the full Hilbert space, convert to MPS, and recover it again with `mps2vec`.

In [5]:
B = basis(L=L, N=Nhalf, k=0, p=1)
H = trans_inv_operator(spin((1.0, "xx"), (1.0, "yy"), (0.5, "zz")), 2, B)
vals, vecs = eigen(Hermitian(H))
v_sector = vecs[:, 1]

ψ_full = sector_to_full(v_sector, B)
ψ_mps = vec2mps(ψ_full, siteinds(B.B, length(B)))
v_back = mps2vec(ψ_mps, B)

println("ground-state recovery error = ", norm(v_back - v_sector))

ground-state recovery error = 3.608224830031759e-16


## Interpretation

So the correct interpretation is:

- `mps2vec(psi, B)` is a **sector-coordinate extractor** for states already in `B`.
- It is **not** a generic projector from the full Hilbert space to `B`.

That matches how it is used in the existing AKLT tests: those MPS are already constructed to lie in the target symmetry sector.